# 02. Processes

**Tier:** Deep. **Why you'd use this:** Heavy CPU work does not overlap well under the GIL. Separate processes can use more than one core, if the function and its data can be pickled.

**After this lesson you can:** choose a process pool when CPU work should use more cores; map a picklable function across workers; explain why lambdas fail under spawn.
About 25 minutes. **Uses:** [Threads](01-threads-lesson.ipynb)


## Quick review

1. What kind of work do threads help most with?
2. What does the GIL serialize?

<details><summary>Answers</summary>

Waiting (I/O). Pure Python bytecode on one object graph in a process.
</details>


## The idea

Processes do not share memory by default. Arguments and return values cross a boundary (pickle on the default start methods). That isolation prevents many races and also means you pay serialisation cost. Use a pool when many similar CPU jobs should run on several cores. Keep payloads small and picklable.

Processes are separate workshops. Handing work across the street means packing a box (pickle) and unpacking the answer. Nobody can bump into your counter by accident, which removes whole classes of races. The price is startup and serialisation. Tiny jobs spend more time in transit than in work. Large CPU jobs pay the tax once and win on wall clock.

<details><summary>Explain more</summary>

On macOS and Windows the start method is often `spawn`: workers re-import a module instead of inheriting memory. Functions defined only in a notebook cell are not importable, so demos import `square` from `cpu_work.py`. Lambdas and nested functions usually will not pickle.
</details>


## See it work

Time a CPU-ish function with one worker versus two.


In [1]:
import time
from concurrent.futures import ProcessPoolExecutor

from cpu_work import busy


def run(workers):
    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=workers) as pool:
        list(pool.map(busy, [120_000, 120_000, 120_000, 120_000]))
    return time.perf_counter() - start


one = run(1)
two = run(2)
print(one)
print(two)
print(two < one)

0.07761508302064613
0.0723798330000136
True


## Mental model

| situation | processes help? | why |
|---|---|---|
| numeric loop over chunks | yes | each process has its own GIL |
| waiting on sockets | rarely best | threads/async are lighter |
| huge unpicklable sockets | no | cannot send the object |
| one giant shared array without care | careful | need shared memory APIs later |

`ProcessPoolExecutor.map` preserves input order in the returned iterable. `submit` returns futures you can wait on in completion order if you prefer. Always consider what crosses the boundary: pure data structures pickle; lambdas and local functions often do not. Scripts need `if __name__ == "__main__"` under spawn.

Pools shine when each task runs for tens of milliseconds or more of CPU. Below that, prefer batching work inside the worker so pickling happens once per batch. Measure with `time.perf_counter` around the whole map, not a single call.

Prefer naming the worker module after the work (`cpu_work`) so imports stay obvious in reviews.


## Check yourself

Answer these from the example above before you run the cell.


1. Why can `cpu_work.square` run in a worker when a notebook-local `def` often cannot under spawn?
2. What does `map` guarantee about result order?
3. When would two process workers be slower than one?

<details><summary>Answers</summary>

1. Workers import the real module by name. 2. Input order. 3. When tasks are tinier than pickling and process startup.
</details>


In [2]:
from concurrent.futures import ProcessPoolExecutor

from cpu_work import square

with ProcessPoolExecutor(max_workers=2) as pool:
    print(list(pool.map(square, [1, 2, 3])))

[1, 4, 9]


<details><summary>Check your answers</summary>

`[1, 4, 9]`. Each worker runs `square` in another process; `map` returns results in input order.
</details>


## Why it works this way

A process has its own interpreter and memory. Starting a pool runs worker processes that import your module and call the target function on pickled arguments. Results come back pickled too. Isolation means a crash in one worker need not corrupt the parent's heap, and the GIL in each process is separate, so CPU-bound Python can use multiple cores. Cost: process start is heavier than a thread; pickling large objects dominates if you send megabytes per task. Design reason for `ProcessPoolExecutor`: keep the API close to thread pools while moving work off the parent's GIL.

On some platforms the start method is `spawn`, which re-imports the main module. Guard entry points with `if __name__ == "__main__"` in scripts. Notebooks import helpers from a real `.py` file so workers can find them. Deep dive: [The GIL and free-threading](../../internals/12-gil/01-gil-lesson.ipynb) for why processes help CPU when threads do not.

Trace of a map: parent pickles each item, workers compute, parent unpickles results in input order for `map`. Evidence: a CPU-ish function over several chunks finishes faster with two workers than with one when the work outweighs pickling. If the function is tiny, pool overhead can lose.

Pool speedups depend on real work beating pickle cost. Timing cells that compare one worker to two are tagged `volatile` when a busy CI host can blur the gap.

Cancellation is not the issue here; isolation is. Threads share objects by pointer; processes share only what you send. That single design choice explains both the safety win and the pickle tax.


In [3]:
import multiprocessing as mp

print(mp.get_start_method())

spawn


## When to use it

Use processes for CPU-bound Python. Use threads or async for waiting. Avoid huge pickle payloads; put large data in shared memory or files if needed later. Do not assume every object pickles. Do not forget to shut down pools in long-running apps.

Pools shine when each task runs for tens of milliseconds or more of CPU. Below that, prefer batching work inside the worker so pickling happens once per batch. Measure with `time.perf_counter` around the whole map, not a single call.


## Common mistake

Passing a lambda to a process pool and expecting it to run like a thread pool.


In [4]:
import subprocess
import sys
from pathlib import Path

script = Path("_proc_lambda_demo.py")
script.write_text(
    "\n".join(
        [
            "from concurrent.futures import ProcessPoolExecutor",
            "",
            'if __name__ == "__main__":',
            "    with ProcessPoolExecutor(max_workers=1) as pool:",
            "        print(list(pool.map(lambda value: value + 1, [1])))",
            "",
        ]
    ),
    encoding="utf-8",
)
result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
print(result.returncode != 0)
script.unlink()

True


Lambdas do not pickle. Import a top-level function from a module instead.


In [5]:
from concurrent.futures import ProcessPoolExecutor

from cpu_work import square

with ProcessPoolExecutor(max_workers=1) as pool:
    print(list(pool.map(square, [1, 2])))

[1, 4]


## Try it

Change the worked example to four inputs but `max_workers=4`. Does wall time drop further on your machine? When would it stop helping?


## Recap

- Processes isolate memory and can use multiple cores for CPU Python.
- Arguments and results must pickle.
- Import worker functions from a real module in notebooks.

**Next:** [practice](02-processes-practice.ipynb)
